#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set confounders
confounders = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10','f11']
input_dim = len(confounders)

#### helpers

In [ ]:
def get_nuisance_params(configs, nuisance):
    row = configs.loc[configs["nuisance"] == nuisance].iloc[0]

    return dict(
        hidden_dim=int(row["hidden_dim"]),
        learning_rate=float(row["lr"]),
        weight_decay=float(row["weight_decay"]),
        batch_size=int(row["batch_size"]),
        max_epochs=50,
        patience=5)

In [ ]:
def train_binary_response(model, train_loader, val_loader, device, lr=3e-4, 
                          weight_decay=1e-5, max_epochs=50, patience=5, seed=0, pos_weight=None):
    """ Expects loaders that yield (x, t, y). """
    set_seed(seed)

    
    # init
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # weighted criterion
    if pos_weight is None:
        criterion = nn.BCEWithLogitsLoss(reduction="mean")
    else:
        pw = torch.tensor([pos_weight], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw, reduction="mean")

    # early stopping
    best_state, best_val, patience_left = None, float("inf"), patience

    # loop over epochs
    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, n_train = 0.0, 0

        # progress
        with tqdm(train_loader, desc=f"Epoch {epoch}/{max_epochs}", leave=False) as pbar:
            for x, _, y in pbar:

                # forward pass
                x, y = x.to(device), y.to(device)
                logits = model(x)
                
                # backward pass
                loss = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # progress
                bs = y.size(0)
                running_loss += loss.item() * bs
                n_train += bs
                pbar.set_postfix(loss=running_loss / n_train)

        # validation loop
        model.eval()
        running_loss_val, n_val = 0.0, 0
        with torch.no_grad():
            for x_val, _, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)
                logits_val = model(x_val)
                loss_val = criterion(logits_val, y_val) 
                
                # progress
                bs_val = y_val.size(0)
                running_loss_val += loss_val.item() * bs_val
                n_val += bs_val
        val_loss = running_loss_val / max(n_val, 1)

        # progress
        tqdm.write(f"Epoch {epoch:02d} | val_loss={val_loss:.4f}")
        
        # early stopping
        if val_loss + 1e-6 < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    # reset best state
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {"val_loss": best_val}

In [ ]:
def compute_pos_weight(df, outcome_col="Y"):
    n_pos = df[outcome_col].sum()
    n_neg = len(df) - n_pos
    return float(n_neg / max(n_pos, 1))

In [ ]:
def train_nuisance_model(nuisance, train_df, val_df, params, seed):
    
    # propensity score
    if nuisance == "e":
        # loaders
        train_loader, val_loader = make_nuisance_loaders(train_df, val_df, confounders, params["batch_size"])

        # init model
        model = ClassificationHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)

        # train model
        model, info = train_propensity(
            model,
            train_loader,
            val_loader,
            device,
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"],
            max_epochs=params["max_epochs"],
            patience=params["patience"],
            seed=seed)

        return model, info, "prop_model.pt"

    # response surfaces
    if nuisance in ["m0", "m1"]:

        # split training data
        treatment_value = 0 if nuisance == "m0" else 1
        train = train_df[train_df["T"] == treatment_value]
        val = val_df[val_df["T"] == treatment_value]

        # loaders
        pos_weight = compute_pos_weight(train, outcome_col="Y")
        train_loader, val_loader = make_nuisance_loaders(train, val, confounders, params["batch_size"])
    
        # init model
        model = ClassificationHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)

        # train model
        model, info = train_binary_response(
            model,
            train_loader,
            val_loader,
            device,
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"],
            max_epochs=params["max_epochs"],
            patience=params["patience"],
            seed=seed,
            pos_weight=pos_weight)
    
        filename = f"mu{treatment_value}_model.pt"
        return model, info, filename

    raise ValueError(f"Unknown nuisance: {nuisance}")

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/criteo_nuisances.csv', index_col=0)

In [ ]:
# set checkpoint dir
out_dir = f'./chkpts/nuisances/'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# loop over nuisances
for nuisance in ["e", "m0", "m1"]:

    # get hyperparameter settings
    params = get_nuisance_params(configs, nuisance)

    # loop over seeds
    for seed in range(5):

        # track progress
        print(f" -> Nuisance {nuisance}, seed {seed}")
        set_seed(seed)

        # set output dir
        ckpt_dir = Path(out_dir) / f"seed_{seed}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)

        # get training data
        train_df = pd.read_csv(ROOT / "experiments" / "criteo" / "data" / "datasets" / f"seed_{seed}" / "train_1.csv")
        val_df = pd.read_csv(ROOT / "experiments" / "criteo" / "data" / "datasets" / f"seed_{seed}" / "val_1.csv")

        # train nuisance model
        set_seed(seed)
        model, info, filename = train_nuisance_model(nuisance=nuisance,train_df=train_df,val_df=val_df,params=params,seed=seed)

        # checkpoint
        torch.save(model.state_dict(), ckpt_dir / filename)